# OpenPII 1.5M — unified English PII dataset builder

This single notebook builds `unified_pii_dataset.csv`, `pii_dataset_summary.csv`,
`pii_dataset_summary.xlsx`, `label2id.json`, and `id2label.json` from:

- [OpenPII 1.5M](https://huggingface.co/datasets/ai4privacy/pii-masking-openpii-1.5m) (English rows only)
- [Kiji PII](https://huggingface.co/datasets/DataikuNLP/kiji-pii-training-data) (English rows only)
- [Gretel PII](https://huggingface.co/datasets/gretelai/gretel-pii-masking-en-v1)
- [Nemotron-PII](https://huggingface.co/datasets/nvidia/Nemotron-PII) (English)
- Saudi/MENA entity resources: [Arabic name pairs](https://huggingface.co/datasets/Sayedyousef/arabic-name-pairs), [Muslim names](https://huggingface.co/datasets/wpacademy/muslim-names-dataset), [ANETAC](https://arxiv.org/abs/1907.03110), [RCRC traffic intersections](https://opendata.rcrc.gov.sa/explore/dataset/traffic-intersections-by-main-street-and-cross-street-2024/export/?flg=en-gb), [National Address cities API](https://api.address.gov.sa/cities), [SAMA licensed banks](https://www.sama.gov.sa/en-us/supervision/licenseentities/pages/licensedbanks.aspx), and CC0 Wikidata organisation/city labels.

Design rules: output text is English and Latin-script only; only safe label mappings are retained;
ambiguous source labels are dropped and audited. OpenPII official English validation is preserved as
project final test and never enters training or augmentation. Every span is checked against
`text[start:end]` before it is written. Saudi identifiers are synthesized only where a reliable format
is implemented; uncertain ID, licence, passport, and tax formats are skipped and reported.

**Runtime note.** A full run streams more than 1.8 million source rows and can take hours. Set `PII_RUN_MODE=smoke` for a small offline structural test. The live source schema snapshot below was verified on 2026-08-12; runtime probing reports schema drift without silently changing mappings.

In [ ]:
# Install only missing runtime packages in the current Jupyter kernel.
import importlib.util, os, subprocess, sys

# Keep Hugging Face locks/cache in a notebook-writable location by default. Override
# PII_CACHE_DIR to place the (potentially large) cache on another volume.
os.environ.setdefault("HF_HOME", os.getenv("PII_CACHE_DIR", os.path.join(os.getcwd(), ".pii_hf_cache")))

_required = {
    "datasets": "datasets>=4.0",
    "pandas": "pandas>=2.0",
    "openpyxl": "openpyxl>=3.1",
    "requests": "requests>=2.31",
    "tqdm": "tqdm>=4.66",
}
_missing = [spec for module, spec in _required.items() if importlib.util.find_spec(module) is None]
if _missing and os.getenv("PII_SKIP_INSTALL", "0") != "1":
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *_missing])
elif _missing:
    raise ImportError(f"Missing packages with PII_SKIP_INSTALL=1: {_missing}")

In [ ]:
import ast
import csv
import hashlib
import io
import json
import math
import random
import re
import unicodedata
from collections import Counter, defaultdict
from dataclasses import dataclass, field
from pathlib import Path
from typing import Any, Iterable, Iterator

import pandas as pd
import requests
from datasets import load_dataset, load_dataset_builder
from IPython.display import display
from tqdm.auto import tqdm

SEED = int(os.getenv("PII_SEED", "20260812"))
RUN_MODE = os.getenv("PII_RUN_MODE", "full").strip().lower()  # full | smoke
if RUN_MODE not in {"full", "smoke"}:
    raise ValueError("PII_RUN_MODE must be 'full' or 'smoke'")

OUTPUT_DIR = Path(os.getenv("PII_OUTPUT_DIR", str(Path.cwd()))).resolve()
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
CSV_PATH = OUTPUT_DIR / "unified_pii_dataset.csv"
XLSX_PATH = OUTPUT_DIR / "pii_dataset_summary.xlsx"
SUMMARY_CSV_PATH = OUTPUT_DIR / "pii_dataset_summary.csv"
LABEL2ID_PATH = OUTPUT_DIR / "label2id.json"
ID2LABEL_PATH = OUTPUT_DIR / "id2label.json"
HF_TOKEN = os.getenv("HF_TOKEN") or None
NATIONAL_ADDRESS_API_KEY = os.getenv("NATIONAL_ADDRESS_API_KEY") or None
HF_DATASETS_CACHE = Path(os.environ["HF_HOME"]) / "datasets"
HF_DATASETS_CACHE.mkdir(parents=True, exist_ok=True)

VALIDATION_FRACTION = 0.10
ENABLE_NEAR_DEDUP = True
NEAR_DUP_HAMMING = 2
MAX_LSH_BUCKET = 128
CONTEXT_RESERVOIR_SIZE = int(os.getenv("PII_CONTEXT_RESERVOIR_SIZE", "8000" if RUN_MODE == "full" else "40"))
REPLACEMENT_BUDGET = int(os.getenv("PII_REPLACEMENT_BUDGET", "5000" if RUN_MODE == "full" else "20"))
MAX_AUGMENT_ROWS = int(os.getenv("PII_MAX_AUGMENT_ROWS", "60000" if RUN_MODE == "full" else "80"))
SAUDI_ROW_TARGET_MIN = int(os.getenv("PII_SAUDI_ROW_TARGET_MIN", "40000" if RUN_MODE == "full" else "0"))
SAUDI_ROW_TARGET_MAX = int(os.getenv("PII_SAUDI_ROW_TARGET_MAX", "60000" if RUN_MODE == "full" else "1000000"))
_max_source_rows_env = os.getenv("PII_MAX_SOURCE_ROWS")
MAX_SOURCE_ROWS_PER_SPLIT = (int(_max_source_rows_env) if _max_source_rows_env else
                             (None if RUN_MODE == "full" else 20))
REQUEST_TIMEOUT = 90
RNG = random.Random(SEED)

UNIVERSAL_LABELS = (
    "ACCOUNTNUM", "AGE", "AMOUNT", "BANKNAME", "BUILDINGNUM", "CITY", "COUNTRY",
    "CREDITCARDNUMBER", "CURRENCY", "DATE", "DRIVERLICENSENUM", "EMAIL", "GENDER",
    "GIVENNAME", "IDCARDNUM", "IPV4", "ORGANISATION", "PASSPORTNUM", "PASSWORD", "SALARY", "SEX",
    "SOCIALNUM", "STREET", "SURNAME", "TAXNUM", "TELEPHONENUM", "TIME", "TIMEZONE",
    "TITLE", "URL", "WEIGHT", "ZIPCODE",
)
DELETED_LABELS = {"USERNAME", "ALLERGIES", "HEIGHT"}

GOAL_ROWS = [
    ("ACCOUNTNUM", 15_000, 5_000), ("AGE", 60_000, 0), ("AMOUNT", 10_000, 0),
    ("BANKNAME", 10_000, 5_000), ("BUILDINGNUM", 60_000, 5_000), ("CITY", 90_000, 10_000),
    ("COUNTRY", 10_000, 2_000), ("CREDITCARDNUMBER", 40_000, 0), ("CURRENCY", 10_000, 3_000),
    ("DATE", 140_000, 0), ("DRIVERLICENSENUM", 40_000, 5_000), ("EMAIL", 90_000, 0),
    ("GENDER", 35_000, 0), ("GIVENNAME", 160_000, 15_000), ("IDCARDNUM", 50_000, 8_000),
    ("IPV4", 10_000, 0), ("ORGANISATION", 20_000, 8_000), ("PASSPORTNUM", 30_000, 5_000),
    ("PASSWORD", 0, 0), ("SALARY", 10_000, 0), ("SEX", 25_000, 0), ("SOCIALNUM", 30_000, 0),
    ("STREET", 70_000, 10_000), ("SURNAME", 140_000, 15_000), ("TAXNUM", 35_000, 5_000),
    ("TELEPHONENUM", 70_000, 5_000), ("TIME", 5_000, 0), ("TIMEZONE", 5_000, 1_000),
    ("TITLE", 80_000, 0), ("URL", 10_000, 0), ("WEIGHT", 5_000, 0),
    ("ZIPCODE", 55_000, 5_000),
]
GOALS = {label: {"total": total, "saudi": saudi} for label, total, saudi in GOAL_ROWS}
GOAL_DF = pd.DataFrame(GOAL_ROWS, columns=["Label", "Goal total", "Saudi goal"])
display(GOAL_DF)

## 1. Inspect source schemas before mapping

Hugging Face cards expose some serialized annotations as strings even when their semantic content is a list. The adapters below therefore parse both JSON and Python-literal forms. No mapping is inferred from a field name alone.

In [ ]:
SOURCE_REGISTRY = {
    "OpenPII": {
        "dataset": "ai4privacy/pii-masking-openpii-1.5m",
        "splits": ("train", "validation"),
        "text": "source_text", "annotation": "privacy_mask", "language": "language",
        "license": "CC-BY-4.0",
    },
    "Kiji": {
        "dataset": "DataikuNLP/kiji-pii-training-data",
        "splits": ("train", "test"),
        "text": "text", "annotation": "privacy_mask", "language": "language",
        "license": "Apache-2.0",
    },
    "Gretel": {
        "dataset": "gretelai/gretel-pii-masking-en-v1",
        "splits": ("train", "validation", "test"),
        "text": "text", "annotation": "entities", "language": None,
        "license": "Apache-2.0",
    },
    "Nemotron": {
        "dataset": "nvidia/Nemotron-PII",
        "splits": ("train", "test"),
        "text": "text", "annotation": "spans", "language": None,
        "license": "CC-BY-4.0",
    },
}

SCHEMA_SNAPSHOT = pd.DataFrame([
    ("OpenPII", "train, validation", "source_text:string; privacy_mask:list; language:string", "1,636,375", "19"),
    ("Kiji", "train, test", "text:string; privacy_mask:list; language:string", "51,495", "26"),
    ("Gretel", "train, validation, test", "text:string; entities:string", "60,000", "40+"),
    ("Nemotron", "train, test", "text:string; spans:string in viewer", "200,000", "55+"),
], columns=["source_dataset", "live_splits_2026_08_12", "relevant_features", "rows", "native_labels"])
display(SCHEMA_SNAPSHOT)

def probe_live_schemas() -> pd.DataFrame:
    rows = []
    if RUN_MODE == "smoke":
        return SCHEMA_SNAPSHOT.assign(runtime_probe="skipped in smoke mode")
    for source, spec in SOURCE_REGISTRY.items():
        try:
            builder = load_dataset_builder(spec["dataset"], token=HF_TOKEN, cache_dir=str(HF_DATASETS_CACHE))
            split_names = ", ".join(sorted((builder.info.splits or {}).keys()))
            features = builder.info.features
            fields = ", ".join(f"{k}:{v}" for k, v in (features or {}).items())
            rows.append({"source_dataset": source, "runtime_splits": split_names,
                         "runtime_features": fields, "probe_status": "ok"})
        except Exception as exc:
            rows.append({"source_dataset": source, "runtime_splits": "", "runtime_features": "",
                         "probe_status": f"warning: {type(exc).__name__}: {exc}"})
    return pd.DataFrame(rows)

LIVE_SCHEMA_AUDIT = probe_live_schemas()
display(LIVE_SCHEMA_AUDIT)

## 2. Universal label mapping

Mapping decisions are explicit. Generic `name`, full `address`, medical identifiers, routing numbers, licence plates, composite date-times, states, credentials, and other unsupported/ambiguous concepts are dropped. `USERNAME`, `ALLERGIES`, and `HEIGHT` are required deletions.

In [ ]:
OPENPII_MAP = {label: label for label in UNIVERSAL_LABELS}
OPENPII_MAP.update({label: None for label in DELETED_LABELS})

KIJI_MAP = {
    "FIRSTNAME": "GIVENNAME", "SURNAME": "SURNAME", "CITY": "CITY",
    "BUILDINGNUM": "BUILDINGNUM", "STREET": "STREET", "ZIP": "ZIPCODE",
    "STATE": None, "COUNTRY": "COUNTRY", "COMPANYNAME": "ORGANISATION",
    "PHONENUMBER": "TELEPHONENUM", "EMAIL": "EMAIL", "DATEOFBIRTH": "DATE",
    "URL": "URL", "DRIVERLICENSENUM": "DRIVERLICENSENUM", "SSN": "SOCIALNUM",
    "SECURITYTOKEN": None, "PASSPORTID": "PASSPORTNUM", "IBAN": "ACCOUNTNUM",
    "PASSWORD": "PASSWORD", "NATIONALID": "IDCARDNUM", "LICENSEPLATENUM": None,
    "TAXNUM": "TAXNUM", "IDCARDNUM": "IDCARDNUM", "AGE": "AGE",
    "CREDITCARDNUMBER": "CREDITCARDNUMBER", "USERNAME": None,
}

SAFE_GENERIC_MAP = {
    "first_name": "GIVENNAME", "given_name": "GIVENNAME", "last_name": "SURNAME",
    "surname": "SURNAME", "company_name": "ORGANISATION", "organization_name": "ORGANISATION",
    "organisation_name": "ORGANISATION", "organization": "ORGANISATION", "organisation": "ORGANISATION",
    "street_address": "STREET", "street": "STREET",
    "phone_number": "TELEPHONENUM", "telephone_number": "TELEPHONENUM", "phone": "TELEPHONENUM",
    "national_id": "IDCARDNUM", "national_id_number": "IDCARDNUM", "id_card_number": "IDCARDNUM",
    "ssn": "SOCIALNUM", "social_security_number": "SOCIALNUM", "postal_code": "ZIPCODE",
    "postcode": "ZIPCODE", "zip_code": "ZIPCODE", "account_number": "ACCOUNTNUM",
    "bank_account_number": "ACCOUNTNUM", "iban": "ACCOUNTNUM", "credit_card_number": "CREDITCARDNUMBER",
    "credit_debit_card": "CREDITCARDNUMBER",
    "date_of_birth": "DATE", "birth_date": "DATE", "date": "DATE", "time": "TIME",
    "city": "CITY", "country": "COUNTRY", "tax_id": "TAXNUM", "tax_number": "TAXNUM",
    "url": "URL", "email": "EMAIL", "email_address": "EMAIL", "ipv4": "IPV4",
    "age": "AGE", "gender": "GENDER", "sex": "SEX", "title": "TITLE", "weight": "WEIGHT",
    "password": "PASSWORD",
    "salary": "SALARY", "amount": "AMOUNT", "currency": "CURRENCY", "timezone": "TIMEZONE",
    "time_zone": "TIMEZONE", "bank_name": "BANKNAME", "building_number": "BUILDINGNUM",
    "building_num": "BUILDINGNUM", "passport_number": "PASSPORTNUM", "passport_id": "PASSPORTNUM",
    "driver_license_number": "DRIVERLICENSENUM", "drivers_license_number": "DRIVERLICENSENUM",
    "driving_license_number": "DRIVERLICENSENUM", "taxpayer_identification_number": "TAXNUM",
}

DROP_REASONS = {
    "username": "required deletion", "user_name": "required deletion", "allergies": "required deletion",
    "height": "required deletion", "state": "no safe universal state/province label",
    "securitytoken": "credential/token has no universal label", "security_token": "credential/token has no universal label",
    "licenseplatenum": "vehicle plate is not a driver licence",
    "license_plate": "vehicle plate is not a driver licence", "medical_record_number": "medical identifier is not a generic ID card",
    "customer_id": "customer ID is ambiguous", "employee_id": "employee ID is ambiguous",
    "name": "full/generic name cannot be split safely", "full_name": "full name cannot be split safely",
    "address": "full/generic address cannot be narrowed safely", "device_identifier": "unsupported identifier",
    "bank_routing_number": "routing number is not an account number", "routing_number": "routing number is not an account number",
    "date_time": "composite date-time cannot map to one label", "unique_identifier": "identifier type is ambiguous",
    "biometric_identifier": "unsupported biometric identifier", "certificate_license_number": "licence type is ambiguous",
    "vehicle_identifier": "unsupported vehicle identifier", "coordinate": "coordinates have no universal label",
    "api_key": "credential has no universal label", "ipv6": "taxonomy contains IPV4 only",
    "health_plan_beneficiary_number": "health-plan ID is not safely generic", "swift_bic": "BIC is not a bank name",
    "cvv": "CVV is not a credit-card number", "pin": "credential has no universal label",
    "nationality": "nationality is not necessarily a country mention", "ip_address": "IP version is ambiguous",
}

SOURCE_MAPS = {"OpenPII": OPENPII_MAP, "Kiji": KIJI_MAP, "Gretel": {}, "Nemotron": {}}

def native_key(label: Any) -> str:
    value = str(label or "").strip()
    value = re.sub(r"^[BI]-", "", value, flags=re.IGNORECASE)
    return re.sub(r"[\s\-]+", "_", value).lower()

def mapping_decision(source: str, native_label: Any) -> tuple[str | None, str, str]:
    raw = str(native_label or "").strip()
    source_map = SOURCE_MAPS.get(source, {})
    if raw in source_map:
        target = source_map[raw]
        if target:
            return target, "mapped", "explicit source mapping"
        reason = DROP_REASONS.get(native_key(raw), "explicit unsafe/unsupported source label")
        return None, "dropped", reason
    upper = re.sub(r"^[BI]-", "", raw.upper())
    if upper in UNIVERSAL_LABELS:
        return upper, "mapped", "identity mapping"
    if upper in DELETED_LABELS:
        return None, "dropped", "required deletion"
    key = native_key(raw)
    if key in SAFE_GENERIC_MAP:
        return SAFE_GENERIC_MAP[key], "mapped", "explicit safe alias"
    if key in DROP_REASONS:
        return None, "dropped", DROP_REASONS[key]
    return None, "dropped", "unmapped; no safe mapping (never guessed)"

def mapping_specification() -> pd.DataFrame:
    rows = []
    for source, source_map in SOURCE_MAPS.items():
        for native, target in source_map.items():
            _, status, reason = mapping_decision(source, native)
            rows.append((source, native, target or "", status, reason))
    for native in sorted(set(SAFE_GENERIC_MAP) | set(DROP_REASONS)):
        target, status, reason = mapping_decision("Gretel/Nemotron", native)
        rows.append(("Gretel/Nemotron", native, target or "", status, reason))
    return (pd.DataFrame(rows, columns=["source_dataset", "native_label", "universal_label", "status", "reason"])
            .drop_duplicates().sort_values(["source_dataset", "native_label"]).reset_index(drop=True))

MAPPING_SPEC = mapping_specification()
display(MAPPING_SPEC)

In [ ]:
@dataclass
class MappingAudit:
    events: Counter = field(default_factory=Counter)

    def add(self, source: str, native: Any, target: str | None, status: str, reason: str, count: int = 1) -> None:
        self.events[(source, str(native or ""), target or "", status, reason)] += count

    def decide(self, source: str, native: Any, count: int = 1) -> str | None:
        target, status, reason = mapping_decision(source, native)
        self.add(source, native, target, status, reason, count)
        return target

    def dataframe(self) -> pd.DataFrame:
        observed = pd.DataFrame([
            (*key, count) for key, count in self.events.items()
        ], columns=["source_dataset", "native_label", "universal_label", "status", "reason", "observed_spans"])
        spec = MAPPING_SPEC.copy()
        spec["observed_spans"] = 0
        out = pd.concat([spec, observed], ignore_index=True)
        out = (out.groupby(["source_dataset", "native_label", "universal_label", "status", "reason"], dropna=False,
                           as_index=False)["observed_spans"].sum()
               .sort_values(["source_dataset", "status", "native_label"]).reset_index(drop=True))
        return out

def parse_serialized(value: Any) -> list:
    if value is None or (isinstance(value, float) and math.isnan(value)):
        return []
    if isinstance(value, list):
        return value
    if isinstance(value, tuple):
        return list(value)
    if not isinstance(value, str):
        return []
    value = value.strip()
    if not value:
        return []
    for parser in (json.loads, ast.literal_eval):
        try:
            result = parser(value)
            return result if isinstance(result, list) else []
        except Exception:
            pass
    return []

def repair_offset(text: str, start: Any, end: Any, value: Any) -> tuple[int, int, str] | None:
    try:
        start_i, end_i = int(start), int(end)
    except Exception:
        return None
    value_s = "" if value is None else str(value)
    if 0 <= start_i < end_i <= len(text):
        actual = text[start_i:end_i]
        if not value_s or actual == value_s:
            return start_i, end_i, actual
    if value_s and 0 <= start_i <= end_i < len(text) and text[start_i:end_i + 1] == value_s:
        return start_i, end_i + 1, value_s
    if value_s:
        lo, hi = max(0, start_i - 12), min(len(text), end_i + 12 + len(value_s))
        candidates = [m.start() for m in re.finditer(re.escape(value_s), text[lo:hi])]
        if len(candidates) == 1:
            fixed_start = lo + candidates[0]
            return fixed_start, fixed_start + len(value_s), value_s
    return None

def drop_overlaps(spans: list[dict], source: str, audit: MappingAudit) -> list[dict]:
    kept, last_end = [], -1
    for span in sorted(spans, key=lambda s: (s["start"], -(s["end"] - s["start"]))):
        if span["start"] < last_end:
            audit.add(source, span.get("_native_label", span["label"]), span["label"], "dropped_overlap",
                      "overlapping mapped span")
            continue
        span.pop("_native_label", None)
        kept.append(span)
        last_end = span["end"]
    return kept

def normalize_direct_spans(text: str, raw_spans: Any, source: str, audit: MappingAudit) -> list[dict]:
    output = []
    for item in parse_serialized(raw_spans):
        if not isinstance(item, dict):
            continue
        native = item.get("label") or item.get("type")
        target, status, reason = mapping_decision(source, native)
        if not target:
            audit.add(source, native, target, status, reason)
            continue
        value = item.get("value", item.get("text"))
        fixed = repair_offset(text, item.get("start"), item.get("end"), value)
        if not fixed:
            audit.add(source, native, target, "dropped_invalid_offset", "offset/value could not be validated")
            continue
        start, end, actual = fixed
        audit.add(source, native, target, "mapped", reason)
        output.append({"start": start, "end": end, "label": target, "value": actual,
                       "is_saudi": False, "_native_label": str(native or "")})
    return drop_overlaps(output, source, audit)

def normalize_gretel_entities(text: str, raw_entities: Any, audit: MappingAudit) -> list[dict]:
    candidates = []
    for item in parse_serialized(raw_entities):
        if not isinstance(item, dict):
            continue
        value = str(item.get("entity") or "")
        native_types = item.get("types") or item.get("type") or []
        if isinstance(native_types, str):
            native_types = [native_types]
        decisions = [(*mapping_decision("Gretel", native), native) for native in native_types]
        mapped = {target for target, status, reason, native in decisions if target}
        if len(mapped) != 1:
            for target, status, reason, native in decisions:
                audit.add("Gretel", native, target, "dropped_type_conflict" if len(mapped) > 1 else status,
                          "multiple safe targets for one entity" if len(mapped) > 1 else reason)
            continue
        target = next(iter(mapped))
        for secondary_target, secondary_status, secondary_reason, secondary_native in decisions:
            if secondary_target is None:
                audit.add("Gretel", secondary_native, None, secondary_status, secondary_reason)
        native_for_target = next(native for mapped_target, _, _, native in decisions if mapped_target == target)
        if not value:
            audit.add("Gretel", native_for_target, target, "dropped_missing_value", "entity value is empty")
            continue
        candidates.append((value, target, native_for_target))

    occupied, output = [], []
    for value, target, native in sorted(candidates, key=lambda x: -len(x[0])):
        matches = list(re.finditer(re.escape(value), text))
        accepted = 0
        for match in matches:
            start, end = match.span()
            if any(start < b and end > a for a, b in occupied):
                continue
            output.append({"start": start, "end": end, "label": target, "value": text[start:end],
                           "is_saudi": False, "_native_label": str(native)})
            occupied.append((start, end))
            accepted += 1
        if accepted:
            target2, _, reason = mapping_decision("Gretel", native)
            audit.add("Gretel", native, target2, "mapped", reason, accepted)
        else:
            audit.add("Gretel", native, target, "dropped_value_not_found", "entity string not found in text")
    return drop_overlaps(output, "Gretel", audit)

def validate_row(row: dict) -> None:
    text = row["text"]
    assert not contains_non_latin_script(text), "non-Latin script found in English-only dataset"
    spans = row["spans"]
    previous_end = -1
    for span in spans:
        assert span["label"] in UNIVERSAL_LABELS, span
        assert 0 <= span["start"] < span["end"] <= len(text), span
        assert text[span["start"]:span["end"]] == span["value"], (span, text)
        assert span["start"] >= previous_end, "overlapping spans"
        previous_end = span["end"]
    assert row["labels_present"] == sorted({s["label"] for s in spans})
    assert row["generation_type"] in {"original", "replacement", "synthetic"}
    assert row["final_split"] in {"train", "validation", "test"}
    assert bool(row["is_saudi"]) == any(bool(s.get("is_saudi")) for s in spans)

def stable_int(value: str, salt: str = "") -> int:
    return int.from_bytes(hashlib.blake2b(f"{SEED}|{salt}|{value}".encode("utf-8"), digest_size=8).digest(), "big")

def assigned_train_split(identity: str) -> str:
    return "validation" if stable_int(identity, "validation") % 10_000 < int(VALIDATION_FRACTION * 10_000) else "train"

def stable_row_id(source: str, source_split: str, uid: Any, text: str) -> str:
    identity = str(uid) if uid not in (None, "") else hashlib.sha256(text.encode("utf-8")).hexdigest()[:24]
    return f"{source.lower()}:{source_split}:{identity}"

## 3. Source adapters and split preservation

OpenPII follows the established project convention: official English **train** is deterministically divided into project train/validation, while official English **validation** becomes the untouched project final test. Kiji and Nemotron have no validation split, so a fixed-seed hash assigns 10% of their **train** rows to validation. Gretel's published splits and all source test rows remain intact. Final-test streams are processed first, and no final-test row enters the replacement reservoir or template generation.

In [ ]:
SMOKE_ROWS = {
    ("OpenPII", "validation"): [{"uid": 1, "language": "en", "source_text": "Ms Ada Stone lives at 10 Palm Road in Riyadh.",
        "privacy_mask": [{"start": 3, "end": 6, "label": "GIVENNAME", "value": "Ada"},
                         {"start": 7, "end": 12, "label": "SURNAME", "value": "Stone"},
                         {"start": 22, "end": 24, "label": "BUILDINGNUM", "value": "10"},
                         {"start": 25, "end": 34, "label": "STREET", "value": "Palm Road"},
                         {"start": 38, "end": 44, "label": "CITY", "value": "Riyadh"}]}],
    ("OpenPII", "train"): [{"uid": 2, "language": "en", "source_text": "Email Lina Reed at lina@example.com on 10 May 2026.",
        "privacy_mask": [{"start": 6, "end": 10, "label": "GIVENNAME", "value": "Lina"},
                         {"start": 11, "end": 15, "label": "SURNAME", "value": "Reed"},
                         {"start": 19, "end": 35, "label": "EMAIL", "value": "lina@example.com"},
                         {"start": 39, "end": 50, "label": "DATE", "value": "10 May 2026"}]}],
    ("Kiji", "test"): [{"text": "Call Omar Khan at +1 555 0100.", "language": "English",
        "privacy_mask": [{"start": 5, "end": 9, "label": "FIRSTNAME", "value": "Omar"},
                         {"start": 10, "end": 14, "label": "SURNAME", "value": "Khan"},
                         {"start": 18, "end": 29, "label": "PHONENUMBER", "value": "+1 555 0100"}]}],
    ("Kiji", "train"): [{"text": "North Labs sent IBAN GB29NWBK60161331926819 and password cedar!47.", "language": "English",
        "privacy_mask": [{"start": 0, "end": 10, "label": "COMPANYNAME", "value": "North Labs"},
                         {"start": 21, "end": 43, "label": "IBAN", "value": "GB29NWBK60161331926819"},
                         {"start": 57, "end": 65, "label": "PASSWORD", "value": "cedar!47"}]}],
    ("Gretel", "test"): [{"uid": "g1", "text": "Sara Moss uses 198.51.100.8.",
        "entities": "[{'entity':'Sara','types':['first_name']},{'entity':'Moss','types':['last_name']},{'entity':'198.51.100.8','types':['ipv4']}]"}],
    ("Gretel", "validation"): [],
    ("Gretel", "train"): [{"uid": "g2", "text": "Acme LLC account 12345 was opened in Jeddah.",
        "entities": "[{'entity':'Acme LLC','types':['company_name']},{'entity':'12345','types':['account_number']},{'entity':'Jeddah','types':['city']}]"}],
    ("Nemotron", "test"): [{"uid": "n1", "text": "Tax number 300000000000003 belongs to Demo Co.",
        "spans": "[{'start':11,'end':26,'text':'300000000000003','label':'tax_id'},{'start':38,'end':45,'text':'Demo Co','label':'company_name'}]"}],
    ("Nemotron", "train"): [{"uid": "n2", "text": "A username quiet_orchid and password BlueCedar!47 were entered.",
        "spans": "[{'start':11,'end':23,'text':'quiet_orchid','label':'user_name'},{'start':37,'end':49,'text':'BlueCedar!47','label':'password'}]"}],
}

SOURCE_SCHEDULE = [
    # OpenPII official English validation is the untouched project final test.
    ("OpenPII", "validation"), ("Kiji", "test"), ("Gretel", "test"), ("Nemotron", "test"),
    ("Gretel", "validation"),
    ("OpenPII", "train"), ("Kiji", "train"), ("Gretel", "train"), ("Nemotron", "train"),
]

FILTER_AUDIT = Counter()

_DISALLOWED_SCRIPT_RE = re.compile(
    r"[\u0370-\u03ff\u0400-\u052f\u0590-\u05ff\u0600-\u08ff\u0900-\u0dff"
    r"\u0e00-\u0fff\u1100-\u11ff\u3040-\u30ff\u3400-\u9fff\uac00-\ud7af]"
)

def contains_non_latin_script(text: str) -> bool:
    """Reject Arabic and other non-Latin scripts while retaining Latin diacritics."""
    if _DISALLOWED_SCRIPT_RE.search(text):
        return True
    return any(
        unicodedata.category(ch).startswith("L")
        and ord(ch) > 127
        and "LATIN" not in unicodedata.name(ch, "")
        for ch in text
    )

def raw_stream(source: str, source_split: str) -> Iterable[dict]:
    if RUN_MODE == "smoke":
        yield from SMOKE_ROWS.get((source, source_split), [])
        return
    spec = SOURCE_REGISTRY[source]
    ds = load_dataset(spec["dataset"], split=source_split, streaming=True, token=HF_TOKEN,
                      cache_dir=str(HF_DATASETS_CACHE))
    for index, item in enumerate(ds):
        if MAX_SOURCE_ROWS_PER_SPLIT is not None and index >= MAX_SOURCE_ROWS_PER_SPLIT:
            break
        yield item

def is_english(source: str, item: dict) -> bool:
    if source == "OpenPII":
        return str(item.get("language", "")).strip().lower() in {"en", "english"}
    if source == "Kiji":
        return str(item.get("language", "")).strip().lower() == "english"
    return True  # Gretel and Nemotron cards specify English.

def normalize_source_item(source: str, source_split: str, item: dict, audit: MappingAudit) -> dict | None:
    if not is_english(source, item):
        FILTER_AUDIT[(source, "non_english_language_metadata")] += 1
        return None
    text_field = SOURCE_REGISTRY[source]["text"]
    text = str(item.get(text_field) or "")
    if not text.strip():
        FILTER_AUDIT[(source, "empty_text")] += 1
        return None
    if contains_non_latin_script(text):
        FILTER_AUDIT[(source, "non_latin_script")] += 1
        return None
    if source == "Gretel":
        spans = normalize_gretel_entities(text, item.get("entities"), audit)
    else:
        spans = normalize_direct_spans(text, item.get(SOURCE_REGISTRY[source]["annotation"]), source, audit)
    if not spans:
        return None
    uid = item.get("uid")
    if source == "OpenPII" and source_split == "validation":
        final_split = "test"
    elif source == "OpenPII" and source_split == "train":
        final_split = assigned_train_split(str(uid or text))
    else:
        final_split = source_split
    if source_split == "train" and source in {"Kiji", "Nemotron"}:
        final_split = assigned_train_split(str(uid or text))
    row = {
        "id": stable_row_id(source, source_split, uid, text),
        "text": text,
        "spans": spans,
        "labels_present": sorted({s["label"] for s in spans}),
        "source_dataset": source,
        "source_split": source_split,
        "final_split": final_split,
        "is_saudi": False,
        "generation_type": "original",
    }
    validate_row(row)
    return row

## 4. Streaming deduplication and online audit counters

Exact comparison normalises Unicode and whitespace without altering the output text. Near-duplicate comparison uses a span-masked structural SimHash with four LSH bands; this catches repeated templates with changed entity values while keeping memory bounded. A single intentional replacement derived from a distinct original context is exempt from the structural check (but never from exact deduplication), because entity replacement is a required augmentation method and each reservoir context is used at most once.

In [ ]:
def normalized_text(text: str) -> str:
    return re.sub(r"\s+", " ", unicodedata.normalize("NFKC", text)).strip().casefold()

def structural_text(text: str, spans: list[dict]) -> str:
    parts, cursor = [], 0
    for span in spans:
        parts.append(text[cursor:span["start"]])
        parts.append(f" <{span['label']}> ")
        cursor = span["end"]
    parts.append(text[cursor:])
    return normalized_text("".join(parts))

def simhash64(text: str) -> int:
    tokens = re.findall(r"[\w<>]+", text, flags=re.UNICODE)
    features = tokens + [f"{a}|{b}" for a, b in zip(tokens, tokens[1:])]
    if not features:
        return 0
    vector = [0] * 64
    for feature in features:
        value = int.from_bytes(hashlib.blake2b(feature.encode("utf-8"), digest_size=8).digest(), "big")
        for bit in range(64):
            vector[bit] += 1 if value & (1 << bit) else -1
    return sum((1 << bit) for bit, score in enumerate(vector) if score >= 0)

class Deduplicator:
    def __init__(self) -> None:
        self.exact_hashes: set[bytes] = set()
        self.structure_hashes: set[bytes] = set()
        self.signatures: set[int] = set()
        self.buckets: dict[tuple[int, int], list[int]] = defaultdict(list)
        self.counts = Counter()

    def check_and_add(self, text: str, spans: list[dict], allow_replacement_structure: bool = False) -> tuple[bool, str | None]:
        exact = hashlib.blake2b(normalized_text(text).encode("utf-8"), digest_size=12).digest()
        if exact in self.exact_hashes:
            self.counts["exact"] += 1
            return False, "exact"
        structure = structural_text(text, spans)
        structure_hash = hashlib.blake2b(structure.encode("utf-8"), digest_size=12).digest()
        word_count = len(structure.split())
        if allow_replacement_structure:
            signature = None
        elif ENABLE_NEAR_DEDUP and word_count >= 7:
            if structure_hash in self.structure_hashes:
                self.counts["same_structure"] += 1
                return False, "same_structure"
            signature = simhash64(structure)
            candidates = set()
            for band in range(4):
                key = (band, (signature >> (16 * band)) & 0xFFFF)
                candidates.update(self.buckets.get(key, ()))
            if any((signature ^ other).bit_count() <= NEAR_DUP_HAMMING for other in candidates):
                self.counts["near"] += 1
                return False, "near"
        else:
            signature = None
        self.exact_hashes.add(exact)
        self.structure_hashes.add(structure_hash)
        if signature is not None and signature not in self.signatures:
            self.signatures.add(signature)
            for band in range(4):
                key = (band, (signature >> (16 * band)) & 0xFFFF)
                bucket = self.buckets[key]
                if len(bucket) < MAX_LSH_BUCKET:
                    bucket.append(signature)
        return True, None

@dataclass
class Stats:
    total_rows: int = 0
    total_spans: int = 0
    total_saudi_rows: int = 0
    total_saudi_spans: int = 0
    rows_by_split: Counter = field(default_factory=Counter)
    spans_by_split: Counter = field(default_factory=Counter)
    rows_by_source: Counter = field(default_factory=Counter)
    spans_by_source: Counter = field(default_factory=Counter)
    saudi_rows_by_source: Counter = field(default_factory=Counter)
    saudi_spans_by_source: Counter = field(default_factory=Counter)
    rows_by_generation: Counter = field(default_factory=Counter)
    spans_by_generation: Counter = field(default_factory=Counter)
    label_rows: Counter = field(default_factory=Counter)
    label_spans: Counter = field(default_factory=Counter)
    saudi_label_rows: Counter = field(default_factory=Counter)
    saudi_label_spans: Counter = field(default_factory=Counter)
    label_source_rows: Counter = field(default_factory=Counter)
    label_source_spans: Counter = field(default_factory=Counter)
    label_split_rows: Counter = field(default_factory=Counter)
    label_split_spans: Counter = field(default_factory=Counter)
    label_generation_rows: Counter = field(default_factory=Counter)
    label_generation_spans: Counter = field(default_factory=Counter)

    def add(self, row: dict) -> None:
        spans = row["spans"]
        labels = {s["label"] for s in spans}
        saudi_labels = {s["label"] for s in spans if s.get("is_saudi")}
        saudi_span_count = sum(bool(s.get("is_saudi")) for s in spans)
        self.total_rows += 1; self.total_spans += len(spans)
        self.rows_by_split[row["final_split"]] += 1; self.spans_by_split[row["final_split"]] += len(spans)
        self.rows_by_source[row["source_dataset"]] += 1; self.spans_by_source[row["source_dataset"]] += len(spans)
        self.rows_by_generation[row["generation_type"]] += 1; self.spans_by_generation[row["generation_type"]] += len(spans)
        if row["is_saudi"]:
            self.total_saudi_rows += 1; self.total_saudi_spans += saudi_span_count
            self.saudi_rows_by_source[row["source_dataset"]] += 1
            self.saudi_spans_by_source[row["source_dataset"]] += saudi_span_count
        for label in labels:
            self.label_rows[label] += 1
            self.label_source_rows[(label, row["source_dataset"])] += 1
            self.label_split_rows[(label, row["final_split"])] += 1
            self.label_generation_rows[(label, row["generation_type"])] += 1
        for label in saudi_labels:
            self.saudi_label_rows[label] += 1
        for span in spans:
            label = span["label"]
            self.label_spans[label] += 1
            self.label_source_spans[(label, row["source_dataset"])] += 1
            self.label_split_spans[(label, row["final_split"])] += 1
            self.label_generation_spans[(label, row["generation_type"])] += 1
            if span.get("is_saudi"):
                self.saudi_label_spans[label] += 1

CSV_COLUMNS = ["id", "text", "spans", "labels_present", "source_dataset", "source_split",
               "final_split", "is_saudi", "generation_type"]

class CsvSink:
    def __init__(self, path: Path) -> None:
        self.path = path
        self.tmp_path = path.with_suffix(path.suffix + ".tmp")
        self.handle = None
        self.writer = None

    def __enter__(self):
        self.handle = self.tmp_path.open("w", encoding="utf-8", newline="")
        self.writer = csv.DictWriter(self.handle, fieldnames=CSV_COLUMNS)
        self.writer.writeheader()
        return self

    def write(self, row: dict) -> None:
        validate_row(row)
        serial = dict(row)
        serial["spans"] = json.dumps(row["spans"], ensure_ascii=False, separators=(",", ":"))
        serial["labels_present"] = json.dumps(row["labels_present"], ensure_ascii=False, separators=(",", ":"))
        serial["is_saudi"] = bool(row["is_saudi"])
        self.writer.writerow(serial)

    def __exit__(self, exc_type, exc, tb):
        if self.handle:
            self.handle.close()
        if exc_type is None:
            self.tmp_path.replace(self.path)
        return False

class ContextReservoir:
    def __init__(self, capacity: int, rng: random.Random) -> None:
        self.capacity, self.rng, self.seen = capacity, rng, 0
        self.rows: list[dict] = []

    def consider(self, row: dict) -> None:
        if row["generation_type"] != "original" or row["final_split"] == "test":
            return
        if not ({"GIVENNAME", "SURNAME", "STREET", "CITY", "ORGANISATION", "BANKNAME",
                 "COUNTRY", "CURRENCY", "TIMEZONE"} & set(row["labels_present"])):
            return
        self.seen += 1
        compact = {**row, "spans": [dict(s) for s in row["spans"]]}
        if len(self.rows) < self.capacity:
            self.rows.append(compact)
        else:
            index = self.rng.randrange(self.seen)
            if index < self.capacity:
                self.rows[index] = compact

## 5. Saudi/MENA public resources and leakage-aware entity pools

`NATIONAL_ADDRESS_API_KEY` is optional. Without it, the notebook combines the API documentation sample, a curated public-city fallback, and Wikidata labels. SAMA and RCRC fetchers keep official URLs and log fallbacks. ANETAC is logged but not used directly because public mirrors do not consistently expose its Person/Location/Organisation class; treating every transliteration as a given name, surname, city, or organisation would violate the no-guess rule. LiStreet is likewise not fetched automatically because a machine-readable licence was not established here.

In [ ]:
RESOURCE_AUDIT: list[dict] = []
SPLIT_EXCEPTIONS: list[dict] = []

def audit_resource(name: str, url: str, status: str, records: int, notes: str) -> None:
    RESOURCE_AUDIT.append({"resource": name, "url": url, "status": status, "records": records, "notes": notes})

def clean_entity(value: Any, max_len: int = 120) -> str | None:
    text = re.sub(r"\s+", " ", str(value or "")).strip(" \t\r\n,;|")
    if not text or len(text) > max_len or any(ch in text for ch in "\x00\x01\x02"):
        return None
    return text

def fetch_json(url: str, **kwargs) -> Any:
    headers = {"User-Agent": "OpenPII-unified-builder/1.0 (research dataset preparation)"}
    response = requests.get(url, timeout=REQUEST_TIMEOUT, headers=headers, **kwargs)
    response.raise_for_status()
    return response.json()

def load_given_name_records() -> list[tuple[str, str | None, str | None]]:
    fallback = [(f"fallback-{i}", en, ar) for i, (en, ar) in enumerate([
        ("Mohammed", "Mohammed"),
        ("Amina", "أمينة"), ("Noura", "نورة"), ("Salman", "سلمان"), ("Faisal", "فيصل"),
        ("Maha", "مها"), ("Rayan", "ريان"), ("Lama", "لمى"), ("Ziyad", "زياد"),
    ])]
    if RUN_MODE == "smoke":
        audit_resource("Muslim names", SOURCE_REGISTRY.get("names", {}).get("dataset", "wpacademy/muslim-names-dataset"),
                       "smoke fallback", len(fallback), "offline smoke data")
        return fallback
    url = "https://huggingface.co/datasets/Sayedyousef/arabic-name-pairs/resolve/main/dict_FINAL.json"
    transliterations = {}
    try:
        transliterations = fetch_json(url)
        audit_resource("Arabic name pairs", url, "loaded", len(transliterations),
                       "used only to cross-check/transliterate rows already identified as given names")
    except Exception as exc:
        audit_resource("Arabic name pairs", url, "fallback", 0, repr(exc))
    records = []
    try:
        ds = load_dataset("wpacademy/muslim-names-dataset", split="train", streaming=True, token=HF_TOKEN,
                          cache_dir=str(HF_DATASETS_CACHE))
        for index, row in enumerate(ds):
            if index >= 30_000:
                break
            en = clean_entity(row.get("english_name"), 40)
            ar = clean_entity(row.get("arabic_name"), 40)
            if en and not ar:
                ar = clean_entity(transliterations.get(en.casefold()), 40)
            if en or ar:
                records.append((f"muslim-name-{index}", en, ar))
        audit_resource("Muslim names", "https://huggingface.co/datasets/wpacademy/muslim-names-dataset",
                       "loaded", len(records), "given-name vocabulary; CC0 dataset card")
    except Exception as exc:
        audit_resource("Muslim names", "https://huggingface.co/datasets/wpacademy/muslim-names-dataset",
                       "fallback", len(fallback), repr(exc))
        records = fallback
    return records or fallback

def fictional_surname_records() -> list[tuple[str, str, str]]:
    left = ["Nur", "Sahil", "Wadi", "Najm", "Rawd", "Badr", "Rimal", "Fajr", "Safa", "Manar",
            "Razan", "Taym", "Lujain", "Hattan", "Dima", "Ruba", "Mazin", "Tala", "Nahil", "Aseel"]
    right = ["ani", "awi", "iri", "aniyah", "ouni", "afi", "ari", "uli", "ami", "adi",
             "qani", "hiri", "sari", "zani", "rami", "lani", "tari", "fari", "miri", "nawi"]
    ar_left = ["نور", "ساحل", "وادي", "نجم", "روض", "بدر", "رمال", "فجر", "صفا", "منار",
               "رزان", "تيم", "لجين", "هتان", "ديما", "ربى", "مازن", "تالا", "نهيل", "أصيل"]
    ar_right = ["اني", "اوي", "يري", "انية", "وني", "افي", "اري", "ولي", "امي", "ادي",
                "قاني", "هيري", "ساري", "زاني", "رامي", "لاني", "تاري", "فاري", "ميري", "ناوي"]
    rows = []
    for i, a in enumerate(left):
        for j, b in enumerate(right):
            rows.append((f"fictional-family-{i}-{j}", f"Al-{a}{b}", f"ال{ar_left[i]}{ar_right[j]}"))
    audit_resource("Compositional fictional family names", "generated locally", "generated", len(rows),
                   "fictional combinations; not personal records; no punctuation-only variants")
    return rows

def english_saudi_surname_records() -> list[tuple[str, str, None]]:
    """Latin-script public/common family names plus clearly generated family-name forms."""
    public_names = [
        "Al Harbi", "Al Qahtani", "Al Otaibi", "Al Mutairi", "Al Dosari", "Al Shammari",
        "Al Ghamdi", "Al Zahrani", "Al Shehri", "Al Anazi", "Al Subaie", "Al Malki",
        "Al Amri", "Al Juhani", "Al Balawi", "Al Rashidi", "Al Sulami", "Al Hazmi",
        "Al Qarni", "Al Asiri", "Al Harthi", "Al Thaqafi", "Al Fifi", "Al Yamani",
        "Al Tamimi", "Al Khaldi", "Al Nasser", "Al Saleh", "Al Saud", "Al Faisal",
        "Al Ibrahim", "Al Hamad", "Al Majed", "Al Rashed", "Al Sayegh", "Al Turki",
        "Al Khatib", "Al Faris", "Al Mansour", "Al Hakim", "Al Najjar", "Al Bahri",
    ]
    roots = ["Nur", "Sahil", "Wadi", "Najm", "Rawd", "Badr", "Rimal", "Fajr", "Safa", "Manar",
             "Razan", "Taym", "Lujain", "Hattan", "Dima", "Ruba", "Mazin", "Tala", "Nahil", "Aseel"]
    suffixes = ["ani", "awi", "iri", "ouni", "afi", "ari", "uli", "ami", "adi", "qani",
                "hiri", "sari", "zani", "rami", "lani", "tari", "fari", "miri", "nawi", "dani"]
    rows = [(f"public-family-{i}", name, None) for i, name in enumerate(public_names)]
    rows.extend((f"fictional-family-en-{i}-{j}", f"Al {root}{suffix}", None)
                for i, root in enumerate(roots) for j, suffix in enumerate(suffixes))
    audit_resource("English Saudi family-name vocabulary", "public/common names plus local generation", "loaded", len(rows),
                   "Latin script only; names are not linked to personal records")
    return rows

def load_rcrc_streets() -> list[tuple[str, str | None, str | None]]:
    url = ("https://opendata.rcrc.gov.sa/api/explore/v2.1/catalog/datasets/"
           "traffic-intersections-by-main-street-and-cross-street-2024/exports/csv")
    fallback_values = [
        ("King Abdulaziz Road", "King Abdulaziz Road"),
        ("King Fahd Road", "طريق الملك فهد"), ("King Abdullah Road", "طريق الملك عبدالله"),
        ("Prince Turki Road", "طريق الأمير تركي"), ("Olaya Street", "شارع العليا"),
        ("Al Washm Street", "شارع الوشم"), ("Makkah Road", "طريق مكة"),
        ("Takhassusi Street", "شارع التخصصي"), ("Eastern Ring Road", "الطريق الدائري الشرقي"),
    ]
    fallback = [(f"street-fallback-{i}", en, ar) for i, (en, ar) in enumerate(fallback_values)]
    if RUN_MODE == "smoke":
        audit_resource("RCRC streets", url, "smoke fallback", len(fallback), "offline smoke data")
        return fallback
    try:
        response = requests.get(url, params={"lang": "en", "timezone": "Asia/Riyadh", "use_labels": "true", "delimiter": ","},
                                timeout=REQUEST_TIMEOUT)
        response.raise_for_status()
        frame = pd.read_csv(io.BytesIO(response.content))
        columns = {str(c).strip().casefold(): c for c in frame.columns}
        street_columns = [columns[k] for k in ("main street", "cross street") if k in columns]
        values = []
        for column in street_columns:
            values.extend(frame[column].dropna().astype(str).tolist())
        unique = sorted({v for value in values if (v := clean_entity(value, 100))})
        records = [(f"rcrc-street-{i}", value, None) for i, value in enumerate(unique)] + fallback
        audit_resource("RCRC streets", url, "loaded", len(records), "official public intersection export")
        return records
    except Exception as exc:
        audit_resource("RCRC streets", url, "fallback", len(fallback), repr(exc))
        return fallback

def wikidata_records(kind: str) -> list[tuple[str, str | None, str | None]]:
    if kind == "city":
        query = """SELECT DISTINCT ?item ?en ?ar WHERE {
          ?item wdt:P17 wd:Q851; wdt:P31/wdt:P279* wd:Q515.
          OPTIONAL {?item rdfs:label ?en FILTER(LANG(?en)='en')}
          OPTIONAL {?item rdfs:label ?ar FILTER(LANG(?ar)='ar')}
        } LIMIT 3000"""
    else:
        query = """SELECT DISTINCT ?item ?en ?ar WHERE {
          ?item wdt:P17 wd:Q851; wdt:P31/wdt:P279* wd:Q43229.
          OPTIONAL {?item rdfs:label ?en FILTER(LANG(?en)='en')}
          OPTIONAL {?item rdfs:label ?ar FILTER(LANG(?ar)='ar')}
        } LIMIT 5000"""
    url = "https://query.wikidata.org/sparql"
    if RUN_MODE == "smoke":
        return []
    try:
        data = fetch_json(url, params={"query": query, "format": "json"})
        rows = []
        for binding in data.get("results", {}).get("bindings", []):
            qid = binding.get("item", {}).get("value", "").rsplit("/", 1)[-1]
            en = clean_entity(binding.get("en", {}).get("value"))
            ar = clean_entity(binding.get("ar", {}).get("value"))
            if qid and (en or ar):
                rows.append((qid, en, ar))
        audit_resource(f"Wikidata Saudi {kind} labels", url, "loaded", len(rows), "CC0 labels, country=Saudi Arabia")
        return rows
    except Exception as exc:
        audit_resource(f"Wikidata Saudi {kind} labels", url, "fallback", 0, repr(exc))
        return []

def load_cities() -> list[tuple[str, str | None, str | None]]:
    fallback_pairs = [
        ("Riyadh", "الرياض"), ("Jeddah", "جدة"), ("Makkah", "مكة المكرمة"),
        ("Medina", "المدينة المنورة"), ("Dammam", "الدمام"), ("Khobar", "الخبر"),
        ("Tabuk", "تبوك"), ("Abha", "أبها"), ("Taif", "الطائف"), ("Yanbu", "ينبع"),
        ("Jazan", "جازان"), ("Hail", "حائل"), ("Buraidah", "بريدة"), ("Najran", "نجران"),
        ("Al Hofuf", "الهفوف"), ("Jubail", "الجبيل"),
    ]
    records = [(f"city-fallback-{i}", en, ar) for i, (en, ar) in enumerate(fallback_pairs)]
    url = "https://apina.address.gov.sa/NationalAddress/v3.1/lookup/cities"
    if NATIONAL_ADDRESS_API_KEY and RUN_MODE == "full":
        try:
            by_city_id: dict[str, dict[str, str]] = defaultdict(dict)
            for language in ("E", "A"):
                data = fetch_json(url, params={"format": "JSON", "regionid": -1, "language": language,
                                               "api_key": NATIONAL_ADDRESS_API_KEY})
                for item in data.get("Cities") or []:
                    value = clean_entity(item.get("Name"))
                    if value:
                        by_city_id[str(item.get("Id"))][language] = value
            records.extend((f"national-address-{city_id}", values.get("E"), values.get("A"))
                           for city_id, values in by_city_id.items())
            audit_resource("National Address cities", url, "loaded", len(records), "official API")
        except Exception as exc:
            audit_resource("National Address cities", url, "fallback", len(records), repr(exc))
    else:
        audit_resource("National Address cities", "https://api.address.gov.sa/cities", "documented fallback", len(records),
                       "set NATIONAL_ADDRESS_API_KEY for the official full lookup")
    return records + wikidata_records("city")

SAMA_BANKS = [
    ("Saudi National Bank", "البنك الأهلي السعودي"), ("Al Rajhi Bank", "مصرف الراجحي"),
    ("Riyad Bank", "بنك الرياض"), ("Saudi Awwal Bank", "البنك السعودي الأول"),
    ("Arab National Bank", "البنك العربي الوطني"), ("Alinma Bank", "مصرف الإنماء"),
    ("Banque Saudi Fransi", "البنك السعودي الفرنسي"), ("Saudi Investment Bank", "البنك السعودي للاستثمار"),
    ("Bank AlJazira", "بنك الجزيرة"), ("Bank Albilad", "بنك البلاد"),
    ("Gulf International Bank Saudi Arabia", "بنك الخليج الدولي السعودية"),
    ("D360 Bank", "بنك دال ثلاثمائة وستون"),
]

def official_bank_records() -> list[tuple[str, str, str]]:
    url = "https://www.sama.gov.sa/en-us/supervision/licenseentities/pages/licensedbanks.aspx"
    rows = [(f"sama-bank-{i}", en, ar) for i, (en, ar) in enumerate(SAMA_BANKS)]
    audit_resource("SAMA licensed banks", url, "curated official-page fallback", len(rows),
                   "closed finite vocabulary; page may block automated requests")
    return rows

def fictional_organisation_records() -> list[tuple[str, str, str]]:
    roots_en = ["Horizon", "Oasis", "Lantern", "Palm", "Dune", "Cedar", "Pearl", "Falcon", "Harbor", "Summit",
                "Nakhla", "Sahab", "Rimal", "Rawabi", "Manara", "Najd", "Tuwaiq", "Wadi", "Qamar", "Bayan"]
    roots_ar = ["الأفق", "الواحة", "الفانوس", "النخلة", "الكثبان", "الأرز", "اللؤلؤ", "الصقر", "المرفأ", "القمة",
                "نخلة", "سحاب", "رمال", "الروابي", "المنارة", "نجد", "طويق", "الوادي", "القمر", "البيان"]
    sectors_en = ["Logistics", "Analytics", "Foods", "Consulting", "Mobility", "Design", "Energy", "Health",
                  "Education", "Trading", "Research", "Cloud", "Construction", "Hospitality", "Services"]
    sectors_ar = ["للخدمات اللوجستية", "للتحليلات", "للأغذية", "للاستشارات", "للتنقل", "للتصميم", "للطاقة", "للصحة",
                  "للتعليم", "للتجارة", "للأبحاث", "للحوسبة", "للبناء", "للضيافة", "للخدمات"]
    rows = []
    for i, root in enumerate(roots_en):
        for j, sector in enumerate(sectors_en):
            rows.append((f"fictional-org-{i}-{j}", f"{root} {sector} Cooperative",
                         f"تعاونية {roots_ar[i]} {sectors_ar[j]} التجريبية"))
    audit_resource("Compositional fictional organisations", "generated locally", "generated", len(rows),
                   "semantic root/sector combinations; clearly synthetic Arabic forms")
    return rows

def partition_records(records: list[tuple[str, str | None, str | None]], label: str) -> dict[str, dict[str, list[str]]]:
    pools = {split: {"en": [], "ar": []} for split in ("train", "validation", "test")}
    for identity, en, ar in records:
        # Prefer the public entity string itself as the identity key so duplicate source rows
        # cannot place the same visible entity in different splits. The paired Arabic form follows
        # its English identity (or becomes the key when English is absent).
        visible_identity = normalized_text(en or ar or identity)
        bucket = stable_int(visible_identity, f"entity:{label}") % 100
        split = "train" if bucket < 80 else "validation" if bucket < 90 else "test"
        if en:
            pools[split]["en"].append(en)
        if ar:
            pools[split]["ar"].append(ar)
    for split in pools:
        for language in pools[split]:
            pools[split][language] = sorted(set(pools[split][language]))
    return pools

def shared_closed_pool(values: list[tuple[str, str]], label: str, reason: str) -> dict[str, dict[str, list[str]]]:
    SPLIT_EXCEPTIONS.append({"label": label, "exception": reason, "mitigation": "split-disjoint sentence phrase banks"})
    return {split: {"en": [x[0] for x in values], "ar": [x[1] for x in values]}
            for split in ("train", "validation", "test")}

def build_entity_pools() -> dict[str, dict[str, dict[str, list[str]]]]:
    given = load_given_name_records()
    surname = english_saudi_surname_records()
    streets = load_rcrc_streets()
    cities = load_cities()
    banks = official_bank_records()
    organisations = wikidata_records("organisation") + fictional_organisation_records()
    audit_resource("ANETAC", "https://arxiv.org/abs/1907.03110", "not directly mapped", 79_924,
                   "public mirrors lack reliable class in the distributed rows; Person/Location cannot be narrowed safely")
    audit_resource("LiStreet Saudi addresses", "https://listreet.com/pages/saudi-arabia", "not auto-fetched", 0,
                   "machine-readable licence/access not established; RCRC and National Address used instead")
    pools = {
        "GIVENNAME": partition_records(given, "GIVENNAME"),
        "SURNAME": partition_records(surname, "SURNAME"),
        "STREET": partition_records(streets, "STREET"),
        "CITY": partition_records(cities, "CITY"),
        "ORGANISATION": partition_records(organisations, "ORGANISATION"),
        "BANKNAME": shared_closed_pool([(en, ar) for _, en, ar in banks], "BANKNAME", "official finite bank list"),
        "COUNTRY": shared_closed_pool([("Saudi Arabia", "المملكة العربية السعودية")], "COUNTRY", "closed country vocabulary"),
        "CURRENCY": shared_closed_pool([("Saudi riyal", "ريال سعودي"), ("SAR", "ر.س")], "CURRENCY", "closed currency vocabulary"),
        "TIMEZONE": shared_closed_pool([("Asia/Riyadh", "آسيا/الرياض"), ("UTC+03:00", "غرينتش+03:00")],
                                           "TIMEZONE", "closed Saudi timezone vocabulary"),
    }
    # Small public vocabularies may not populate every split. Share only as an audited last resort.
    for label, split_pools in pools.items():
        if label in {"BANKNAME", "COUNTRY", "CURRENCY", "TIMEZONE"}:
            continue
        all_en = sorted({v for p in split_pools.values() for v in p["en"]})
        all_ar = sorted({v for p in split_pools.values() for v in p["ar"]})
        for split, langs in split_pools.items():
            for lang, all_values in (("en", all_en), ("ar", all_ar)):
                if not langs[lang] and all_values:
                    langs[lang] = all_values
                    SPLIT_EXCEPTIONS.append({"label": label, "exception": f"empty {split}/{lang} identity partition",
                                             "mitigation": "shared fallback entity; split-disjoint sentence phrase banks"})
    return pools

class PoolCycler:
    def __init__(self, pools: dict, rng: random.Random) -> None:
        self.pools = pools
        self.rng = rng
        self.orders: dict[tuple[str, str, str], list[str]] = {}
        self.indices = Counter()

    def next(self, label: str, split: str, language: str) -> str:
        key = (label, split, language)
        if key not in self.orders:
            values = list(self.pools.get(label, {}).get(split, {}).get(language, []))
            if not values:
                raise KeyError(f"No values for {key}")
            local = random.Random(stable_int("|".join(key), "pool-order"))
            local.shuffle(values)
            self.orders[key] = values
        values = self.orders[key]
        index = self.indices[key]
        self.indices[key] += 1
        return values[index % len(values)]

class ValueFactory:
    def __init__(self, pools: dict, seed: int) -> None:
        self.pool = PoolCycler(pools, random.Random(seed))
        self.rng = random.Random(seed)
        self.counters = Counter()

    def _number(self, label: str, split: str) -> int:
        key = (label, split)
        self.counters[key] += 1
        offsets = {"train": 100_000, "validation": 600_000, "test": 800_000}
        return offsets[split] + self.counters[key]

    @staticmethod
    def _iban(check_payload: str) -> str:
        rearranged = check_payload + "281000"  # SA00 -> S=28, A=10, then 00
        check = 98 - (int(rearranged) % 97)
        return f"SA{check:02d}{check_payload}"

    def get(self, label: str, split: str, language: str = "en") -> str:
        if label in self.pool.pools:
            return self.pool.next(label, split, language)
        n = self._number(label, split)
        if label == "ACCOUNTNUM":
            payload = f"{n % 100:02d}{(n * 982451653) % 10**18:018d}"
            return self._iban(payload)
        if label == "TELEPHONENUM":
            operator_codes = (50, 53, 54, 55, 56, 57, 58, 59)
            operator = operator_codes[n % len(operator_codes)]
            subscriber = (n * 7919) % 10_000_000
            return f"+966 {operator} {subscriber // 10_000:03d} {subscriber % 10_000:04d}"
        if label == "BUILDINGNUM":
            return str(1 + (n % 9998))
        if label == "ZIPCODE":
            base = {"train": 11000, "validation": 21000, "test": 31000}[split]
            return str(base + n % 999)
        if label in {"AMOUNT", "SALARY"}:
            value = 500 + ((n * 137) % 50_000)
            return f"{value:,}.00"
        if label == "URL":
            return f"https://example.com/sa/services/{n}"
        raise KeyError(label)

## 6. Replacement and varied synthetic Saudi contexts

Replacement rows retain every untouched original span, mark only replaced spans as Saudi, and inherit the source train/validation split. Synthetic rows use split-specific phrase banks and split-partitioned identities. Closed vocabularies are documented exceptions.

In [ ]:
SAUDI_FOCUS_LABELS = {
    "GIVENNAME", "SURNAME", "STREET", "CITY", "ORGANISATION", "BANKNAME", "IDCARDNUM",
    "ACCOUNTNUM", "TELEPHONENUM", "BUILDINGNUM", "ZIPCODE", "DRIVERLICENSENUM",
    "PASSPORTNUM", "TAXNUM", "COUNTRY", "CURRENCY", "TIMEZONE",
}
SAUDI_SYNTHESIS_SKIPS = {
    "IDCARDNUM": "Saudi ID synthesis skipped because its issuance/checksum rules were not implemented confidently",
    "DRIVERLICENSENUM": "Saudi driver-licence synthesis skipped because a reliable public format was not established",
    "PASSPORTNUM": "Saudi passport synthesis skipped because a reliable public format was not established",
    "TAXNUM": "Saudi tax-number synthesis skipped because validation rules were not implemented confidently",
}
SAUDI_TARGET_LABELS = {label for label in SAUDI_FOCUS_LABELS if GOALS[label]["saudi"] > 0}
SYNTHESIZABLE_SAUDI_LABELS = SAUDI_TARGET_LABELS - set(SAUDI_SYNTHESIS_SKIPS)
REPLACEABLE = {"GIVENNAME", "SURNAME", "STREET", "CITY", "ORGANISATION", "BANKNAME",
               "COUNTRY", "CURRENCY", "TIMEZONE"}

def current_needs(stats: Stats) -> dict[str, int]:
    return {
        label: max(0, GOALS[label]["total"] - stats.label_spans[label],
                   GOALS[label]["saudi"] - stats.saudi_label_spans[label])
        for label in SYNTHESIZABLE_SAUDI_LABELS
    }

def replace_context(row: dict, factory: ValueFactory, needs: dict[str, int], rng: random.Random) -> dict | None:
    eligible = [i for i, span in enumerate(row["spans"])
                if span["label"] in REPLACEABLE and needs.get(span["label"], 0) > 0]
    if not eligible:
        return None
    rng.shuffle(eligible)
    selected = set(eligible[:min(len(eligible), rng.randint(1, 3))])
    language = "en"
    parts, new_spans, cursor = [], [], 0
    for index, span in enumerate(row["spans"]):
        parts.append(row["text"][cursor:span["start"]])
        value = factory.get(span["label"], row["final_split"], language) if index in selected else span["value"]
        start = sum(len(part) for part in parts)
        parts.append(value)
        new_span = {"start": start, "end": start + len(value), "label": span["label"], "value": value,
                    "is_saudi": bool(index in selected)}
        new_spans.append(new_span)
        cursor = span["end"]
    parts.append(row["text"][cursor:])
    text = "".join(parts)
    result = {
        "id": "replacement:" + hashlib.sha256((row["id"] + text).encode("utf-8")).hexdigest()[:28],
        "text": text, "spans": new_spans,
        "labels_present": sorted({s["label"] for s in new_spans}),
        "source_dataset": f"Saudi/MENA replacement:{row['source_dataset']}",
        "source_split": f"derived_from_{row['source_split']}", "final_split": row["final_split"],
        "is_saudi": True, "generation_type": "replacement",
    }
    validate_row(result)
    return result

TRAIN_CONTEXT = {
    "documents": ["onboarding note", "service request", "delivery record", "compliance memo", "customer letter",
                  "permit application", "account review", "supplier form", "case summary", "travel declaration"],
    "purposes": ["identity verification", "address confirmation", "payment reconciliation", "licence renewal",
                 "vendor registration", "benefit enrolment", "shipment release", "audit follow-up", "portal access",
                 "contract activation", "branch transfer", "tax review"],
    "channels": ["secure portal", "branch desk", "registered letter", "mobile workflow", "service kiosk", "support queue"],
    "departments": ["operations", "customer care", "finance", "mobility", "procurement", "compliance", "logistics", "HR"],
    "subjects": ["retail services", "small-business support", "urban mobility", "housing services", "visitor services",
                 "warehouse access", "professional licensing", "digital enrolment", "vendor onboarding", "benefit administration",
                 "cross-city delivery", "branch operations", "workforce services", "facility access", "payment operations",
                 "document custody", "customer recovery", "travel support", "contract servicing", "municipal coordination"],
    "stages": ["initial", "amended", "scheduled", "expedited", "routine", "conditional", "supervised", "final"],
    "routes": ["standard", "assisted", "priority", "cross-branch", "mobile-first", "paper-backed", "dual-review", "self-service"],
}
VALID_CONTEXT = {
    "documents": ["verification docket", "review worksheet", "authorisation brief", "registry extract"],
    "purposes": ["secondary screening", "exception handling", "quality assurance", "record correction"],
    "channels": ["validation console", "review office", "assurance mailbox"],
    "departments": ["assurance", "governance", "records", "quality"],
    "subjects": ["validation services", "exception review", "control testing", "record assurance", "secondary verification",
                 "quality sampling", "policy review", "correction handling"],
    "stages": ["rechecked", "provisional", "reconciled", "quality-gated", "independently reviewed"],
    "routes": ["validation-only", "assurance-led", "exception-controlled", "reviewer-assisted", "secondary-control"],
}
TEST_CONTEXT = {
    "documents": ["evaluation packet", "benchmark notice", "assessment sheet", "holdout memorandum"],
    "purposes": ["independent assessment", "holdout adjudication", "benchmark scoring", "external review"],
    "channels": ["evaluation desk", "assessment portal", "holdout queue"],
    "departments": ["evaluation", "assessment", "independent review", "benchmarking"],
    "subjects": ["holdout services", "benchmark review", "external assessment", "evaluation controls", "independent scoring",
                 "blind adjudication", "test governance", "assessment sampling"],
    "stages": ["held-out", "blind", "sealed", "externally scored", "benchmark-only"],
    "routes": ["holdout-only", "blind-review", "external-evaluation", "sealed-assessment", "benchmark-control"],
}

def context_words(split: str, rng: random.Random) -> dict[str, str]:
    bank = TRAIN_CONTEXT if split == "train" else VALID_CONTEXT if split == "validation" else TEST_CONTEXT
    return {key: rng.choice(values) for key, values in bank.items()}

def render_parts(parts: list[Any]) -> tuple[str, list[dict]]:
    text_parts, spans, cursor = [], [], 0
    for part in parts:
        if isinstance(part, tuple):
            label, value = part
            start = cursor
            text_parts.append(value); cursor += len(value)
            spans.append({"start": start, "end": cursor, "label": label, "value": value, "is_saudi": True})
        else:
            value = str(part)
            text_parts.append(value); cursor += len(value)
    return "".join(text_parts), spans

class SaudiTemplateEngine:
    FAMILIES = {
        "person_address": {"GIVENNAME", "SURNAME", "BUILDINGNUM", "STREET", "CITY", "ZIPCODE", "COUNTRY", "TELEPHONENUM"},
        "bank_transfer": {"GIVENNAME", "SURNAME", "BANKNAME", "ACCOUNTNUM", "CURRENCY", "CITY"},
        "employment": {"GIVENNAME", "SURNAME", "ORGANISATION", "CITY"},
        "delivery": {"BUILDINGNUM", "STREET", "CITY", "ZIPCODE", "TELEPHONENUM"},
        "meeting": {"ORGANISATION", "CITY", "TIMEZONE"},
        "support": {"BANKNAME", "ACCOUNTNUM", "TELEPHONENUM"},
    }

    def __init__(self, factory: ValueFactory, seed: int) -> None:
        self.factory = factory
        self.rng = random.Random(seed)

    def v(self, label: str, split: str, lang: str) -> tuple[str, str]:
        return label, self.factory.get(label, split, lang)

    def make(self, family: str, split: str) -> dict:
        # Saudi/MENA entities and contexts are English/Latin-script only.
        if family not in self.FAMILIES:
            raise KeyError(f"Saudi synthesis family is disabled or unknown: {family}")
        lang = "en"
        c = context_words(split, self.rng)
        v = lambda label: self.v(label, split, lang)
        if family == "person_address":
            parts = ([f"For the {c['purposes']} {c['documents']}, record ", v("GIVENNAME"), " ", v("SURNAME"),
                      " at building ", v("BUILDINGNUM"), ", ", v("STREET"), ", ", v("CITY"), " ", v("ZIPCODE"),
                      ", ", v("COUNTRY"), "; contact ", v("TELEPHONENUM"), f" through the {c['channels']}."] if lang == "en" else
                     [f"لغرض {c['purposes']} في {c['departments']}، سُجل الاسم ", v("GIVENNAME"), " ", v("SURNAME"),
                      " في المبنى ", v("BUILDINGNUM"), "، ", v("STREET"), "، ", v("CITY"), " ", v("ZIPCODE"),
                      "، ", v("COUNTRY"), "، ورقم التواصل ", v("TELEPHONENUM"), "."])
        elif family == "bank_transfer":
            parts = ([f"The {c['departments']} {c['documents']} for ", v("GIVENNAME"), " ", v("SURNAME"), " in ", v("CITY"),
                      " authorises a transfer denominated in ", v("CURRENCY"), " through ", v("BANKNAME"),
                      " to account ", v("ACCOUNTNUM"), f" for {c['purposes']}."] if lang == "en" else
                     [f"يعتمد قسم {c['departments']} طلب ", v("GIVENNAME"), " ", v("SURNAME"), " في ", v("CITY"),
                      " بمبلغ ", v("AMOUNT"), " ", v("CURRENCY"), " عبر ", v("BANKNAME"), " إلى الحساب ", v("ACCOUNTNUM"), "."])
        elif family == "employment":
            parts = ([f"A {c['documents']} from ", v("ORGANISATION"), " confirms that ", v("GIVENNAME"), " ", v("SURNAME"),
                      " works in ", v("CITY"), f" for {c['purposes']} via the {c['channels']}."] if lang == "en" else
                     ["تؤكد ", v("ORGANISATION"), " أن ", v("GIVENNAME"), " ", v("SURNAME"), " يعمل في ", v("CITY"),
                      " براتب شهري ", v("SALARY"), " ", v("CURRENCY"), f" ضمن إجراء {c['purposes']}."])
        elif family == "identity":
            parts = ([f"The {c['documents']} for ", v("GIVENNAME"), " ", v("SURNAME"), " lists fictional demo ID ", v("IDCARDNUM"),
                      ", driver licence ", v("DRIVERLICENSENUM"), ", and passport ", v("PASSPORTNUM"), " for ", v("COUNTRY"),
                      f" during {c['purposes']}."] if lang == "en" else
                     [f"يتضمن {c['documents']} التجريبي باسم ", v("GIVENNAME"), " ", v("SURNAME"), " رقم الهوية ", v("IDCARDNUM"),
                      " ورخصة القيادة ", v("DRIVERLICENSENUM"), " وجواز السفر ", v("PASSPORTNUM"), " في ", v("COUNTRY"), "."])
        elif family == "tax_invoice":
            parts = ([v("ORGANISATION"), f" submitted a {c['documents']} for {c['purposes']}; tax number ", v("TAXNUM"),
                      ", amount ", v("AMOUNT"), " ", v("CURRENCY"), ", bank ", v("BANKNAME"), " and account ", v("ACCOUNTNUM"), "."] if lang == "en" else
                     ["قدمت ", v("ORGANISATION"), f" مستند {c['documents']} لغرض {c['purposes']}؛ الرقم الضريبي ", v("TAXNUM"),
                      " والمبلغ ", v("AMOUNT"), " ", v("CURRENCY"), " لدى ", v("BANKNAME"), " للحساب ", v("ACCOUNTNUM"), "."])
        elif family == "delivery":
            parts = ([f"The {c['departments']} team routed the {c['documents']} to building ", v("BUILDINGNUM"), " on ", v("STREET"),
                      " in ", v("CITY"), " ", v("ZIPCODE"), "; the recipient callback is ", v("TELEPHONENUM"),
                      f" for {c['purposes']}."] if lang == "en" else
                     [f"أرسل قسم {c['departments']} المستند إلى المبنى ", v("BUILDINGNUM"), " في ", v("STREET"), "، ", v("CITY"),
                      " ", v("ZIPCODE"), "، ورقم المستلم ", v("TELEPHONENUM"), f" لغرض {c['purposes']}."])
        elif family == "meeting":
            parts = ([v("ORGANISATION"), f" scheduled a {c['purposes']} session in ", v("CITY"), " using timezone ", v("TIMEZONE"),
                      "; instructions for the ", c["documents"], f" are available through the {c['channels']}."] if lang == "en" else
                     ["حددت ", v("ORGANISATION"), f" جلسة {c['purposes']} في ", v("CITY"), " وفق المنطقة الزمنية ", v("TIMEZONE"),
                      f"، وتعليمات {c['documents']} على ", v("URL"), "."])
        elif family == "travel":
            parts = ([f"For {c['purposes']}, traveller ", v("GIVENNAME"), " ", v("SURNAME"), " presented passport ", v("PASSPORTNUM"),
                      " issued for ", v("COUNTRY"), " at the ", v("CITY"), f" {c['departments']} desk."] if lang == "en" else
                     [f"لإجراء {c['purposes']} قدم المسافر ", v("GIVENNAME"), " ", v("SURNAME"), " جواز السفر ", v("PASSPORTNUM"),
                      " الصادر في ", v("COUNTRY"), " لدى مكتب ", v("CITY"), "."])
        elif family == "permit":
            parts = ([v("ORGANISATION"), f" opened a {c['documents']} in ", v("CITY"), " for ", v("STREET"),
                      "; the fictional ID ", v("IDCARDNUM"), " and driver licence ", v("DRIVERLICENSENUM"),
                      f" support {c['purposes']}."] if lang == "en" else
                     ["فتحت ", v("ORGANISATION"), f" مستند {c['documents']} في ", v("CITY"), " خاصاً بـ ", v("STREET"),
                      " باستخدام الهوية التجريبية ", v("IDCARDNUM"), " والرخصة ", v("DRIVERLICENSENUM"), "."])
        elif family == "support":
            parts = ([f"The {c['channels']} sent a {c['documents']} for ", v("BANKNAME"), " account ", v("ACCOUNTNUM"),
                      "; call ", v("TELEPHONENUM"), f" for {c['purposes']} and quote the service record."] if lang == "en" else
                     [f"أرسل {c['channels']} إشعاراً لحساب ", v("BANKNAME"), " رقم ", v("ACCOUNTNUM"), "؛ للتواصل ",
                      v("TELEPHONENUM"), " أو مراجعة ", v("URL"), f" بشأن {c['purposes']}."])
        else:
            raise KeyError(family)
        if lang == "en":
            parts.extend([f" The {c['stages']} handling note covers {c['subjects']} through the {c['routes']} route."])
        else:
            parts.extend([f" وتخص الملاحظة الإدارية مرحلة {c['stages']} لخدمة {c['subjects']} عبر مسار {c['routes']}."])
        text, spans = render_parts(parts)
        row = {
            "id": "synthetic:" + hashlib.sha256((split + family + text).encode("utf-8")).hexdigest()[:28],
            "text": text, "spans": spans, "labels_present": sorted({s["label"] for s in spans}),
            "source_dataset": "Saudi/MENA synthetic", "source_split": "generated", "final_split": split,
            "is_saudi": True, "generation_type": "synthetic",
        }
        validate_row(row)
        return row

## 7. Build CSV, audit workbook, and final notebook summary

In [ ]:
def source_group_spans(stats: Stats, label: str, source: str) -> int:
    return stats.label_source_spans[(label, source)]

def write_bio_label_maps() -> tuple[dict[str, int], dict[str, str]]:
    bio_labels = ["O"] + [tag for label in UNIVERSAL_LABELS for tag in (f"B-{label}", f"I-{label}")]
    label2id = {label: index for index, label in enumerate(bio_labels)}
    id2label = {str(index): label for label, index in label2id.items()}
    LABEL2ID_PATH.write_text(json.dumps(label2id, indent=2, ensure_ascii=False) + "\n", encoding="utf-8")
    ID2LABEL_PATH.write_text(json.dumps(id2label, indent=2, ensure_ascii=False) + "\n", encoding="utf-8")
    return label2id, id2label

def detailed_label_summary_frame(stats: Stats) -> pd.DataFrame:
    rows = []
    for label in UNIVERSAL_LABELS:
        total_goal, saudi_goal = GOALS[label]["total"], GOALS[label]["saudi"]
        notes = []
        if label == "PASSWORD":
            notes.append("No numeric target supplied; retained from safely mapped source password labels")
        if label in SAUDI_SYNTHESIS_SKIPS:
            notes.append(SAUDI_SYNTHESIS_SKIPS[label])
        if saudi_goal and stats.saudi_label_spans[label] < saudi_goal:
            notes.append("Saudi target below goal")
        if total_goal and stats.label_spans[label] < total_goal:
            notes.append("Total target below goal")
        contributions = {source: stats.label_source_spans[(label, source)]
                         for source in ("OpenPII", "Kiji", "Gretel", "Nemotron")}
        if contributions and max(contributions.values(), default=0) > stats.label_spans[label] * 0.5:
            dominant = max(contributions, key=contributions.get)
            notes.append(f"Target contribution mainly through {dominant}")
        if saudi_goal == 0 and label not in SAUDI_FOCUS_LABELS:
            notes.append("No reliable Saudi-specific augmentation needed")
        rows.append({
            "label": label,
            "total_rows": stats.label_rows[label], "total_spans": stats.label_spans[label],
            "train_rows": stats.label_split_rows[(label, "train")],
            "validation_rows": stats.label_split_rows[(label, "validation")],
            "test_rows": stats.label_split_rows[(label, "test")],
            "saudi_rows": stats.saudi_label_rows[label], "saudi_spans": stats.saudi_label_spans[label],
            "openpii_rows": stats.label_source_rows[(label, "OpenPII")],
            "openpii_spans": stats.label_source_spans[(label, "OpenPII")],
            "kiji_rows": stats.label_source_rows[(label, "Kiji")],
            "kiji_spans": stats.label_source_spans[(label, "Kiji")],
            "gretel_rows": stats.label_source_rows[(label, "Gretel")],
            "gretel_spans": stats.label_source_spans[(label, "Gretel")],
            "nemotron_rows": stats.label_source_rows[(label, "Nemotron")],
            "nemotron_spans": stats.label_source_spans[(label, "Nemotron")],
            "synthetic_rows": stats.label_generation_rows[(label, "synthetic")],
            "synthetic_spans": stats.label_generation_spans[(label, "synthetic")],
            "replacement_rows": stats.label_generation_rows[(label, "replacement")],
            "replacement_spans": stats.label_generation_spans[(label, "replacement")],
            "goal_total_spans": total_goal, "goal_saudi_spans": saudi_goal,
            "total_goal_reached": stats.label_spans[label] >= total_goal,
            "saudi_goal_reached": stats.saudi_label_spans[label] >= saudi_goal,
            "notes": "; ".join(dict.fromkeys(notes)),
        })
    return pd.DataFrame(rows)

def label_summary_frame(stats: Stats) -> pd.DataFrame:
    rows = []
    for label in UNIVERSAL_LABELS:
        synthetic_replacement = sum(
            count for (lab, source), count in stats.label_source_spans.items()
            if lab == label and source.startswith("Saudi/MENA")
        )
        total_goal, saudi_goal = GOALS[label]["total"], GOALS[label]["saudi"]
        reached = stats.label_spans[label] >= total_goal and stats.saudi_label_spans[label] >= saudi_goal
        rows.append({
            "Label": label,
            "Total rows containing label": stats.label_rows[label],
            "Total entity spans": stats.label_spans[label],
            "Saudi rows": stats.saudi_label_rows[label],
            "Saudi spans": stats.saudi_label_spans[label],
            "OpenPII spans": source_group_spans(stats, label, "OpenPII"),
            "Kiji spans": source_group_spans(stats, label, "Kiji"),
            "Gretel spans": source_group_spans(stats, label, "Gretel"),
            "Nemotron spans": source_group_spans(stats, label, "Nemotron"),
            "Synthetic/Replacement spans": synthetic_replacement,
            "Goal total": total_goal, "Saudi goal": saudi_goal,
            "Goal reached?": "Yes" if reached else "No",
        })
    return pd.DataFrame(rows)

def source_summary_frame(stats: Stats) -> pd.DataFrame:
    sources = sorted(stats.rows_by_source)
    return pd.DataFrame([{
        "source_dataset": source, "rows": stats.rows_by_source[source], "spans": stats.spans_by_source[source],
        "Saudi rows": stats.saudi_rows_by_source[source], "Saudi spans": stats.saudi_spans_by_source[source],
    } for source in sources])

def split_summary_frame(stats: Stats) -> pd.DataFrame:
    return pd.DataFrame([{
        "final_split": split, "rows": stats.rows_by_split[split], "spans": stats.spans_by_split[split],
    } for split in ("train", "validation", "test")])

def saudi_summary_frame(stats: Stats) -> pd.DataFrame:
    rows = []
    for source in sorted(stats.rows_by_source):
        rows.append({"source_dataset": source, "Saudi rows": stats.saudi_rows_by_source[source],
                     "Saudi spans": stats.saudi_spans_by_source[source]})
    rows.append({"source_dataset": "ALL", "Saudi rows": stats.total_saudi_rows, "Saudi spans": stats.total_saudi_spans})
    return pd.DataFrame(rows)

def label_matrix(counter: Counter, columns: list[str], column_name: str) -> pd.DataFrame:
    rows = []
    for label in UNIVERSAL_LABELS:
        row = {"Label": label}
        for column in columns:
            row[column] = counter[(label, column)]
        rows.append(row)
    return pd.DataFrame(rows)

def write_summary_workbook(stats: Stats, audit: MappingAudit, dedup: Deduplicator) -> dict[str, pd.DataFrame]:
    mapping = audit.dataframe()
    source_summary = source_summary_frame(stats)
    sheets = {
        "label_summary": label_summary_frame(stats),
        "detailed_label_summary": detailed_label_summary_frame(stats),
        "source_summary": source_summary,
        "split_summary": split_summary_frame(stats),
        "saudi_summary": saudi_summary_frame(stats),
        "label_by_source": label_matrix(stats.label_source_spans, sorted(stats.rows_by_source), "source_dataset"),
        "label_by_split": label_matrix(stats.label_split_spans, ["train", "validation", "test"], "final_split"),
        "mapping_audit": mapping,
        "dropped_labels": mapping[mapping["status"] != "mapped"].reset_index(drop=True),
        "resource_audit": pd.DataFrame(RESOURCE_AUDIT),
        "split_exceptions": pd.DataFrame(SPLIT_EXCEPTIONS),
        "dedup_audit": pd.DataFrame([{"duplicate_type": key, "rows_removed": value}
                                      for key, value in sorted(dedup.counts.items())]),
        "filter_audit": pd.DataFrame([{"source_dataset": source, "filter_reason": reason, "rows_removed": count}
                                      for (source, reason), count in sorted(FILTER_AUDIT.items())]),
        "saudi_synthesis_skips": pd.DataFrame([{"label": label, "reason": reason}
                                                for label, reason in SAUDI_SYNTHESIS_SKIPS.items()]),
    }
    with pd.ExcelWriter(XLSX_PATH, engine="openpyxl") as writer:
        for name, frame in sheets.items():
            frame.to_excel(writer, sheet_name=name[:31], index=False)
            worksheet = writer.book[name[:31]]
            worksheet.freeze_panes = "A2"
            worksheet.auto_filter.ref = worksheet.dimensions
            for column_cells in worksheet.columns:
                width = min(55, max(10, max(len(str(cell.value or "")) for cell in column_cells) + 2))
                worksheet.column_dimensions[column_cells[0].column_letter].width = width
    sheets["detailed_label_summary"].to_csv(SUMMARY_CSV_PATH, index=False, encoding="utf-8")
    write_bio_label_maps()
    return sheets

def add_row(row: dict, sink: CsvSink, stats: Stats, dedup: Deduplicator) -> bool:
    keep, _ = dedup.check_and_add(
        row["text"], row["spans"], allow_replacement_structure=row["generation_type"] == "replacement"
    )
    preserve_openpii_final_test = (row["source_dataset"] == "OpenPII"
                                   and row["source_split"] == "validation"
                                   and row["final_split"] == "test")
    if not keep and preserve_openpii_final_test:
        # Keep every OpenPII official-validation row in the final test. Its already-seen
        # fingerprint still prevents a matching row from entering a later source split.
        reason = _
        dedup.counts[reason] -= 1
        if dedup.counts[reason] <= 0:
            del dedup.counts[reason]
        sink.write(row); stats.add(row)
        return True
    if not keep:
        return False
    sink.write(row); stats.add(row)
    return True

def build_dataset() -> tuple[Stats, MappingAudit, Deduplicator, dict[str, pd.DataFrame]]:
    stats, audit, dedup = Stats(), MappingAudit(), Deduplicator()
    reservoir = ContextReservoir(CONTEXT_RESERVOIR_SIZE, random.Random(SEED + 1))
    pools = build_entity_pools()
    factory = ValueFactory(pools, SEED + 2)

    with CsvSink(CSV_PATH) as sink:
        for source, source_split in SOURCE_SCHEDULE:
            stream = raw_stream(source, source_split)
            for item in tqdm(stream, desc=f"{source}/{source_split}", unit="rows"):
                row = normalize_source_item(source, source_split, item, audit)
                if row is None:
                    continue
                if add_row(row, sink, stats, dedup):
                    reservoir.consider(row)

        # Replacements use only original train/validation contexts.
        contexts = list(reservoir.rows)
        RNG.shuffle(contexts)
        replacements_added = 0
        for context in contexts:
            if replacements_added >= REPLACEMENT_BUDGET:
                break
            needs = current_needs(stats)
            if not any(needs.values()):
                break
            row = replace_context(context, factory, needs, RNG)
            if row is not None and add_row(row, sink, stats, dedup):
                replacements_added += 1

        # Synthetic generation observes the remaining total and Saudi deficits.
        engine = SaudiTemplateEngine(factory, SEED + 3)
        generated, failed_attempts = 0, 0
        split_cycle = (["train"] * 16 + ["validation"] * 2 + ["test"] * 2)
        while generated < MAX_AUGMENT_ROWS and stats.total_saudi_rows < SAUDI_ROW_TARGET_MAX:
            needs = current_needs(stats)
            unmet = {label: value for label, value in needs.items() if value > 0}
            if not unmet and stats.total_saudi_rows >= SAUDI_ROW_TARGET_MIN:
                break
            # First balance the requested Saudi share across labels; then address any remaining
            # total deficit. This prevents a very large total goal (for example GIVENNAME) from
            # starving a smaller Saudi-specific target such as TIMEZONE or TAXNUM.
            def deficit_priority(label: str) -> tuple[float, float, str]:
                saudi_goal = GOALS[label]["saudi"]
                saudi_gap = max(0, saudi_goal - stats.saudi_label_spans[label])
                total_goal = GOALS[label]["total"]
                total_gap = max(0, total_goal - stats.label_spans[label])
                return (saudi_gap / max(1, saudi_goal), total_gap / max(1, total_goal), label)

            if unmet:
                target = max(unmet, key=deficit_priority)
                families = [name for name, labels in engine.FAMILIES.items() if target in labels]
            else:
                families = list(engine.FAMILIES)
            family = RNG.choice(families)
            split = split_cycle[generated % len(split_cycle)]
            row = engine.make(family, split)
            if add_row(row, sink, stats, dedup):
                generated += 1; failed_attempts = 0
            else:
                failed_attempts += 1
                if failed_attempts > 5_000:
                    print("Stopped augmentation after 5,000 consecutive dedup rejections.")
                    break

    sheets = write_summary_workbook(stats, audit, dedup)
    return stats, audit, dedup, sheets

def print_final_summary(stats: Stats, audit: MappingAudit, dedup: Deduplicator, sheets: dict[str, pd.DataFrame]) -> None:
    print(f"Outputs:\n  {CSV_PATH}\n  {SUMMARY_CSV_PATH}\n  {XLSX_PATH}\n  {LABEL2ID_PATH}\n  {ID2LABEL_PATH}")
    print("\nRows by final split:")
    display(sheets["split_summary"])
    print(f"Total rows: {stats.total_rows:,} | total spans: {stats.total_spans:,}")
    print("Rows by generation type:", {kind: stats.rows_by_generation[kind]
                                       for kind in ("original", "replacement", "synthetic")})
    print(f"Saudi rows: {stats.total_saudi_rows:,} | Saudi spans: {stats.total_saudi_spans:,}")
    print("\nCounts for every universal label:")
    display(sheets["label_summary"][["Label", "Total entity spans", "Saudi spans", "Goal total", "Saudi goal", "Goal reached?"]])
    detailed = sheets["detailed_label_summary"]
    failed = detailed[(detailed["goal_total_spans"] > 0) & ~detailed["total_goal_reached"]]
    saudi_failed = detailed[(detailed["goal_saudi_spans"] > 0) & ~detailed["saudi_goal_reached"]]
    print("\nLabels below total target:", ", ".join(failed["label"]) if len(failed) else "none")
    print("Saudi labels below target:", ", ".join(saudi_failed["label"]) if len(saudi_failed) else "none")
    print("\nSource contribution:")
    display(sheets["source_summary"])
    print("Duplicates removed:", dict(dedup.counts), "total=", sum(dedup.counts.values()))
    invalid_spans = sum(count for key, count in audit.events.items() if key[3] == "dropped_invalid_offset")
    print("Invalid spans removed:", invalid_spans)
    dropped = sheets["dropped_labels"]
    observed_dropped = dropped[dropped["observed_spans"] > 0]
    print("\nObserved unmapped/dropped labels:")
    display(observed_dropped[["source_dataset", "native_label", "status", "reason", "observed_spans"]])
    print("\nSaudi synthesis skipped:")
    for label, reason in SAUDI_SYNTHESIS_SKIPS.items():
        print(f"  {label}: {reason}")
    print(f"Offset validation passed online for all {stats.total_rows:,} written rows.")
    print("No exact old OpenPII project split/fingerprint files were available when this notebook was updated. The fixed-seed "
          "train/validation recreation therefore cannot reproduce unavailable historical fingerprints. OpenPII official "
          "validation was assigned only to final test and was never used for training, replacement, reservoir sampling, "
          "or template extraction.")

In [ ]:
STATS, MAPPING_AUDIT, DEDUPLICATOR, SUMMARY_SHEETS = build_dataset()
print_final_summary(STATS, MAPPING_AUDIT, DEDUPLICATOR, SUMMARY_SHEETS)